# 📊 Notebook 02: FAO AQUASTAT Data Extraction

**Objective:** Download and process water resources indicators from FAO AQUASTAT.

**Source:** [AQUASTAT Dissemination Platform](https://data.apps.fao.org/aquastat/)

**Key Variables:**
- SDG 6.4.2 Water Stress (%)
- Total Renewable Water Resources
- Freshwater Withdrawal (Agricultural, Industrial, Municipal)
- Precipitation
- Dependency Ratio

In [ ]:
import sys
sys.path.insert(0, '..')

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import plotly.express as px
import logging
import warnings
warnings.filterwarnings('ignore')
logging.basicConfig(level=logging.INFO)

import config
from src.extractors.aquastat_extractor import AquastatExtractor

plt.style.use('dark_background')
sns.set_theme(style='darkgrid')
print('✅ Setup complete')

## 1. Data Download

In [ ]:
extractor = AquastatExtractor()
extractor.download_data()
print('✅ Data download complete')

## 2. Raw Data Exploration

In [ ]:
raw_df = extractor.load_raw_data()
print(f'Shape: {raw_df.shape}')
print(f'\nColumns: {list(raw_df.columns)}')
print(f'\nUnique variables: {raw_df["Variable"].nunique() if "Variable" in raw_df.columns else "N/A"}')
if 'Variable' in raw_df.columns:
    print(f'\nVariables:\n{raw_df["Variable"].value_counts().head(15)}')
raw_df.head(10)

## 3. Feature Extraction (Long → Wide)

In [ ]:
df = extractor.extract_features(raw_df)
print(f'Processed shape: {df.shape}')
print(f'\nColumns: {list(df.columns)}')
df.head(10)

In [ ]:
# Summary statistics
df.describe().round(2)

## 4. Visualization

In [ ]:
# Water stress over time (top 10 most stressed countries)
stress_col = [c for c in df.columns if 'water_stress' in c.lower()]
if stress_col and 'year' in df.columns and 'country' in df.columns:
    col = stress_col[0]
    
    # Get top stressed countries (latest year)
    latest = df[df['year'] == df['year'].max()]
    top_countries = latest.nlargest(10, col)['country'].tolist()
    
    plot_data = df[df['country'].isin(top_countries)]
    
    fig = px.line(
        plot_data, x='year', y=col, color='country',
        title=f'Water Stress Trends — Top 10 Most Stressed Countries',
        labels={'year': 'Year', col: 'Water Stress (%)'},
        markers=True,
    )
    fig.update_layout(template='plotly_dark', height=500)
    fig.show()

In [ ]:
# Global water withdrawal composition
withdrawal_cols = [c for c in df.columns if 'withdrawal' in c.lower()]
if withdrawal_cols and 'year' in df.columns:
    yearly = df.groupby('year')[withdrawal_cols].sum().reset_index()
    
    fig = px.area(
        yearly, x='year', y=withdrawal_cols,
        title='Global Water Withdrawal by Sector Over Time',
        labels={'value': 'Withdrawal (km³)', 'year': 'Year'},
    )
    fig.update_layout(template='plotly_dark', height=450)
    fig.show()

In [ ]:
# Latest year map — water stress by country
if stress_col and 'country_code' in df.columns:
    col = stress_col[0]
    latest = df[df['year'] == df['year'].max()]
    
    fig = px.choropleth(
        latest, locations='country_code', locationmode='ISO-3',
        color=col, hover_name='country',
        color_continuous_scale='RdYlBu_r',
        title=f'SDG 6.4.2 Water Stress (%) — {int(df["year"].max())}',
    )
    fig.update_layout(
        geo=dict(showframe=False, projection_type='natural earth'),
        template='plotly_dark', height=500,
    )
    fig.show()

## 5. Save Processed Data

In [ ]:
output_path = extractor.save_processed(df)
print(f'✅ Saved to: {output_path}')
print(f'   Shape: {df.shape}')
print(f'   Countries: {df["country"].nunique() if "country" in df.columns else "N/A"}')
print(f'   Years: {df["year"].min()} – {df["year"].max()}' if 'year' in df.columns else '')